In [1]:
import pandas as pd
from readback_guard.config import ROOT, load_config

cfg = load_config()
pairs = pd.read_parquet(ROOT / cfg["data"]["interim_dir"] / "candidate_pairs.parquet")

pool = pairs[pairs["has_value"]].sample(frac=1, random_state=cfg["split"]["seed"])
sheet = pd.concat([
    pool[pool["split"] == "dev"].head(60),
    pool[pool["split"] == "test"].head(120),
])[["pair_id", "split", "atco_text", "pilot_text"]]

for col in ["keep", "atco_callsign", "pilot_callsign", "atco_items", "pilot_items", "verdict", "notes"]:
    sheet[col] = ""

path = ROOT / "labels" / "gold_pairs.csv"
if path.exists():
    print("gold_pairs.csv already exists, not overwriting it")
else:
    sheet.to_csv(path, index=False)
    print("wrote", len(sheet), "rows to", path)

wrote 180 rows to /Users/speedyboi/Documents/atomcamp stuff/read-back guard/labels/gold_pairs.csv


In [2]:
gold = pd.read_csv(ROOT / "labels" / "gold_pairs.csv", dtype=str, keep_default_na=False)

TYPES = {"runway", "frequency", "speed", "squawk", "level", "heading"}
VERDICTS = {"CORRECT", "WRONG_VALUE", "MISSING_ITEM", "WRONG_CALLSIGN"}


def parse_items(cell):
    items = {}
    for part in filter(None, (p.strip() for p in cell.split(";"))):
        kind, value = part.split("=")
        items[kind.strip()] = value.strip()
    return items


labelled = gold[gold["keep"] != ""]
problems = []
for _, row in labelled.iterrows():
    if row["keep"] not in {"y", "n"}:
        problems.append((row["pair_id"], "keep must be y or n"))
    if row["keep"] != "y":
        continue
    if row["verdict"] not in VERDICTS:
        problems.append((row["pair_id"], f"bad verdict {row['verdict']!r}"))
    for side in ("atco_items", "pilot_items"):
        try:
            kinds = set(parse_items(row[side]))
        except ValueError:
            problems.append((row["pair_id"], f"{side} not written as type=value"))
            continue
        if kinds - TYPES:
            problems.append((row["pair_id"], f"unknown type in {side}: {kinds - TYPES}"))

print(problems or "no problems")
kept = labelled[labelled["keep"] == "y"]
print(kept.groupby("split").size())
print(kept["verdict"].value_counts())

no problems
split
dev      52
test    106
dtype: int64
verdict
CORRECT           119
MISSING_ITEM       30
WRONG_CALLSIGN      5
WRONG_VALUE         4
Name: count, dtype: int64
